# E-commerce Sales Intelligence: Performance, Profitability & Business Insights
**Author:** Naveen Kumar K | **Stack:** Python · Pandas · Matplotlib · Seaborn · Power BI  
**Dataset:** User-provided e-commerce transaction data (2024–2025)  
**Last Updated:** 2025-12-31

---
## Executive Summary
This notebook delivers a complete, reproducible end-to-end sales analysis for a
home-goods e-commerce business operating across the United States (2024–2025).

**Key Findings:**
- **$1.89M** in completed net revenue from **8,456 orders** over 24 months
- **49.2% profit margin** — Storage (55.6%) and Decor/Bedding (51.2%) are the most profitable categories
- **64.4% repeat customer rate** — strong retention with 1,318 returning customers
- Discounted orders earn **9.4 pp lower margin** than full-price orders (40.8% vs 50.4%)
- California, New York, and Texas drive **29.3%** of total revenue
- Organic Search is the most efficient channel: **41% of revenue** from **36% of customers**

**Strategic Priorities:**
1. Restrict promotional discounts to high-value RFM segments
2. Re-engage 405 "At Risk" customers before they churn
3. Expand the Storage category (highest margin, only 7.4% revenue share)


## 1. Environment Setup & Imports

In [1]:

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
import warnings

matplotlib.use('Agg')
warnings.filterwarnings('ignore')

# Project paths
BASE = Path('..').resolve()
RAW  = BASE / 'data' / 'raw'
PROC = BASE / 'data' / 'processed'
PBI  = BASE / 'outputs' / 'power_bi'

# Style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans', 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'teal': '#2A9D8F', 'coral': '#E76F51', 'amber': '#E9C46A', 'navy': '#264653', 'grey': '#6B7280'}

print(f'Pandas  : {pd.__version__}')
print(f'NumPy   : {np.__version__}')
print(f'RAW dir : {RAW}')
print(f'PBI dir : {PBI}')


Pandas  : 3.0.2
NumPy   : 2.4.4
RAW dir : /home/claude/ecommerce-sales-intelligence/data/raw
PBI dir : /home/claude/ecommerce-sales-intelligence/outputs/power_bi


## 2. Dataset Discovery & File Inventory

In [2]:

raw_files = list(RAW.glob('*.csv'))
print(f'Raw files found: {len(raw_files)}')
for f in raw_files:
    df_tmp = pd.read_csv(f, nrows=0)
    size_kb = f.stat().st_size / 1024
    wc = sum(1 for _ in open(f)) - 1
    print(f'  {f.name:<30} {wc:>7,} rows | {len(df_tmp.columns):>3} cols | {size_kb:>8.1f} KB')


Raw files found: 4
  customers.csv                    3,600 rows |   4 cols |    107.6 KB
  orders.csv                       9,245 rows |   7 cols |    515.9 KB
  order_items.csv                 15,608 rows |   5 cols |    347.5 KB
  products.csv                        90 rows |   5 cols |      4.0 KB


## 3. Dataset Schema Assessment

In [3]:

orders      = pd.read_csv(RAW/'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(RAW/'order_items.csv')
products    = pd.read_csv(RAW/'products.csv')
customers   = pd.read_csv(RAW/'customers.csv', parse_dates=['signup_date'])

schema = pd.DataFrame([
    dict(Table='orders', Rows=len(orders), Columns=list(orders.columns), Grain='1 row = 1 order', PrimaryKey='order_id'),
    dict(Table='order_items', Rows=len(order_items), Columns=list(order_items.columns), Grain='1 row = 1 product line per order', PrimaryKey='(order_id, product_id)'),
    dict(Table='products', Rows=len(products), Columns=list(products.columns), Grain='1 row = 1 product SKU', PrimaryKey='product_id'),
    dict(Table='customers', Rows=len(customers), Columns=list(customers.columns), Grain='1 row = 1 customer', PrimaryKey='customer_id'),
])
print(schema[['Table','Rows','Grain','PrimaryKey']].to_string(index=False))

print(f'\nDate range: {orders.order_date.min().date()} → {orders.order_date.max().date()}')
print(f'Order statuses: {orders.status.unique().tolist()}')
print(f'Product categories (raw): {products.category.unique().tolist()}')


      Table  Rows                            Grain             PrimaryKey
     orders  9245                  1 row = 1 order               order_id
order_items 15608 1 row = 1 product line per order (order_id, product_id)
   products    90            1 row = 1 product SKU             product_id
  customers  3600               1 row = 1 customer            customer_id

Date range: 2024-01-01 → 2025-12-31
Order statuses: ['completed', 'returned', 'canceled']
Product categories (raw): ['Kitchen', ' Kitchen', 'Kitchen ', 'kitchen', 'KITCHEN', ' Bedding', 'Bedding', 'bedding', 'BEDDING', 'Bath', 'bath', 'Furniture', 'Furniture ', 'furniture', 'FURNITURE', 'Lighting', 'LIGHTING', 'Outdoor', 'OUTDOOR', ' Outdoor', 'Decor', 'decor', 'Storage', 'Storage ']


## 4. Data Quality Assessment

In [4]:

for name, df in [('orders', orders), ('order_items', order_items),
                  ('products', products), ('customers', customers)]:
    nulls = df.isnull().sum()
    dups  = df.duplicated().sum()
    nulls_nonzero = nulls[nulls > 0]
    print(f'--- {name} ---')
    if len(nulls_nonzero):
        print(f'  Missing: {nulls_nonzero.to_dict()}')
    else:
        print(f'  Missing: None')
    print(f'  Duplicates: {dups}')
    print()

# Category inconsistencies
print('Category raw unique values:')
for c in sorted(products.category.unique()):
    print(f'  repr: {repr(c)}')


--- orders ---
  Missing: {'discount_code': 7834}
  Duplicates: 38

--- order_items ---
  Missing: {'unit_price': 199}
  Duplicates: 0

--- products ---
  Missing: None
  Duplicates: 0

--- customers ---
  Missing: None
  Duplicates: 0

Category raw unique values:
  repr: ' Bedding'
  repr: ' Kitchen'
  repr: ' Outdoor'
  repr: 'BEDDING'
  repr: 'Bath'
  repr: 'Bedding'
  repr: 'Decor'
  repr: 'FURNITURE'
  repr: 'Furniture'
  repr: 'Furniture '
  repr: 'KITCHEN'
  repr: 'Kitchen'
  repr: 'Kitchen '
  repr: 'LIGHTING'
  repr: 'Lighting'
  repr: 'OUTDOOR'
  repr: 'Outdoor'
  repr: 'Storage'
  repr: 'Storage '
  repr: 'bath'
  repr: 'bedding'
  repr: 'decor'
  repr: 'furniture'
  repr: 'kitchen'


## 5. Data Cleaning & Transformation

In [5]:

# 5a. Normalize product categories
products_clean = products.copy()
products_clean['category'] = products_clean['category'].str.strip().str.title()
print(f'Clean categories: {sorted(products_clean.category.unique())}')

# 5b. Impute null unit_price from products.list_price
order_items_clean = order_items.copy()
null_mask = order_items_clean['unit_price'].isnull()
print(f'Null unit_prices before impute: {null_mask.sum()}')
price_map = products_clean.set_index('product_id')['list_price']
order_items_clean.loc[null_mask, 'unit_price'] = (
    order_items_clean.loc[null_mask, 'product_id'].map(price_map)
)
print(f'Null unit_prices after impute: {order_items_clean.unit_price.isnull().sum()}')

# 5c. Derive line-level metrics
order_items_clean['gross_revenue'] = order_items_clean['unit_price'] * order_items_clean['quantity']
order_items_clean['net_revenue']   = order_items_clean['gross_revenue'] - order_items_clean['discount_amount']
cost_map = products_clean.set_index('product_id')['unit_cost']
order_items_clean['unit_cost']  = order_items_clean['product_id'].map(cost_map)
order_items_clean['total_cost'] = order_items_clean['unit_cost'] * order_items_clean['quantity']
order_items_clean['profit']     = order_items_clean['net_revenue'] - order_items_clean['total_cost']

print('Line metrics computed: gross_revenue, net_revenue, unit_cost, total_cost, profit')


Clean categories: ['Bath', 'Bedding', 'Decor', 'Furniture', 'Kitchen', 'Lighting', 'Outdoor', 'Storage']
Null unit_prices before impute: 199


Null unit_prices after impute: 0
Line metrics computed: gross_revenue, net_revenue, unit_cost, total_cost, profit


## 6. Feature Engineering

In [6]:

region_map = {
    'CA':'West','OR':'West','WA':'West','NV':'West','AZ':'West','UT':'West','CO':'West',
    'TX':'South','FL':'South','GA':'South','NC':'South','VA':'South','TN':'South',
    'NY':'Northeast','PA':'Northeast','NJ':'Northeast','MA':'Northeast','CT':'Northeast',
    'IL':'Midwest','OH':'Midwest','MI':'Midwest','IN':'Midwest','WI':'Midwest',
    'MN':'Midwest','MO':'Midwest',
}

# Master fact table join
fact = order_items_clean.merge(
    orders[['order_id','customer_id','order_date','status',
            'payment_method','discount_code','shipping_fee']],
    on='order_id', how='left'
).merge(
    products_clean[['product_id','product_name','category']],
    on='product_id', how='left'
).merge(
    customers[['customer_id','signup_date','state','acquisition_channel']],
    on='customer_id', how='left'
)

# Status flags
fact['is_completed'] = fact['status'] == 'completed'
fact['is_returned']  = fact['status'] == 'returned'
fact['is_canceled']  = fact['status'] == 'canceled'

# Date dimensions
fact['order_year']       = fact['order_date'].dt.year
fact['order_quarter']    = fact['order_date'].dt.quarter
fact['order_month_num']  = fact['order_date'].dt.month
fact['order_month_name'] = fact['order_date'].dt.strftime('%b')
fact['order_year_month'] = fact['order_date'].dt.to_period('M').astype(str)
fact['order_weekday']    = fact['order_date'].dt.day_name()

# Discount flag
fact['has_discount'] = (fact['discount_amount'] > 0) | (fact['discount_code'].notna())

# Region
fact['region'] = fact['state'].map(region_map).fillna('Other')

completed = fact[fact['is_completed']].copy()
print(f'Fact table: {len(fact):,} rows')
print(f'Completed:  {len(completed):,} rows across {completed.order_id.nunique():,} orders')


Fact table: 15,668 rows
Completed:  14,339 rows across 8,456 orders


## 7. Executive KPIs

In [7]:

total_revenue   = completed['net_revenue'].sum()
total_profit    = completed['profit'].sum()
total_orders    = completed['order_id'].nunique()
total_customers = completed['customer_id'].nunique()
total_units     = completed['quantity'].sum()
aov             = total_revenue / total_orders
margin          = total_profit / total_revenue * 100

kpis = pd.DataFrame({
    'KPI': ['Total Revenue','Total Profit','Profit Margin','Total Orders',
            'Distinct Customers','Units Sold','Avg Order Value'],
    'Value': [f'${total_revenue:,.0f}', f'${total_profit:,.0f}', f'{margin:.1f}%',
              f'{total_orders:,}', f'{total_customers:,}', f'{total_units:,}', f'${aov:,.2f}']
})
print(kpis.to_string(index=False))


               KPI      Value
     Total Revenue $1,888,956
      Total Profit   $929,083
     Profit Margin      49.2%
      Total Orders      8,456
Distinct Customers      2,047
        Units Sold     17,840
   Avg Order Value    $223.39


## 8. Sales Performance Analysis — Monthly Trends

In [8]:

monthly = (
    completed.groupby('order_year_month')
    .agg(revenue=('net_revenue','sum'), profit=('profit','sum'),
         orders=('order_id','nunique'), units=('quantity','sum'))
    .reset_index().sort_values('order_year_month')
)
monthly['margin_pct'] = monthly['profit'] / monthly['revenue'] * 100
monthly['mom_chg']    = monthly['revenue'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
x = range(len(monthly))
labels = monthly['order_year_month'].tolist()

# Revenue & Profit
axes[0].bar(x, monthly['revenue'], color=COLORS['teal'], alpha=0.85, label='Revenue')
axes[0].bar(x, monthly['profit'],  color=COLORS['navy'], alpha=0.85, label='Profit')
axes[0].set_title('Monthly Revenue & Profit (Completed Orders Only)', fontsize=14, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))
axes[0].legend()

# MoM Change
colors_mom = [COLORS['teal'] if v >= 0 else COLORS['coral'] for v in monthly['mom_chg'].fillna(0)]
axes[1].bar(x, monthly['mom_chg'].fillna(0), color=colors_mom)
axes[1].axhline(0, color='#333', lw=0.8)
axes[1].set_title('Month-over-Month Revenue Change (%)', fontsize=14, fontweight='bold')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'{v:.0f}%'))
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig(PROC/'monthly_trends.png', bbox_inches='tight')
plt.show()
print('Chart saved.')
print(f'Best month:  {monthly.loc[monthly.revenue.idxmax(), "order_year_month"]} (${monthly.revenue.max():,.0f})')
print(f'Worst month: {monthly.loc[monthly.revenue.idxmin(), "order_year_month"]} (${monthly.revenue.min():,.0f})')


Chart saved.
Best month:  2025-11 ($130,442)
Worst month: 2024-01 ($54,452)


## 9. Product & Category Performance

In [9]:

cat_perf = (
    completed.groupby('category')
    .agg(revenue=('net_revenue','sum'), profit=('profit','sum'), units=('quantity','sum'))
    .reset_index().sort_values('revenue', ascending=False)
)
cat_perf['margin_pct'] = cat_perf['profit'] / cat_perf['revenue'] * 100
cat_perf['rev_share']  = cat_perf['revenue'] / cat_perf['revenue'].sum() * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Revenue by Category
bars = axes[0].barh(cat_perf['category'], cat_perf['revenue'],
                    color=COLORS['teal'], edgecolor='white')
axes[0].set_xlabel('Net Revenue ($)')
axes[0].set_title('Revenue by Category', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))
for bar, val in zip(bars, cat_perf['revenue']):
    axes[0].text(bar.get_width() + 2000, bar.get_y() + bar.get_height()/2,
                 f'${val/1e3:.0f}K', va='center', fontsize=9)

# Profit Margin
overall_margin = completed['profit'].sum() / completed['net_revenue'].sum() * 100
bar_colors = [COLORS['teal'] if m >= overall_margin else COLORS['amber'] for m in cat_perf['margin_pct']]
axes[1].barh(cat_perf['category'], cat_perf['margin_pct'], color=bar_colors)
axes[1].axvline(overall_margin, color=COLORS['coral'], linestyle='--', label=f'Avg {overall_margin:.1f}%')
axes[1].set_xlabel('Profit Margin (%)')
axes[1].set_title('Profit Margin by Category', fontweight='bold')
axes[1].legend()
for i, (_, row) in enumerate(cat_perf.iterrows()):
    axes[1].text(row.margin_pct + 0.3, i, f"{row.margin_pct:.1f}%", va='center', fontsize=9)

plt.tight_layout()
plt.savefig(PROC/'category_performance.png', bbox_inches='tight')
plt.show()

print(cat_perf[['category','revenue','profit','margin_pct','rev_share']].to_string(index=False))


 category   revenue    profit  margin_pct  rev_share
Furniture 324741.22 149970.65   46.181587  17.191573
  Kitchen 281060.81 132938.75   47.298928  14.879163
    Decor 278889.57 142757.19   51.187712  14.764219
  Bedding 268245.56 137296.15   51.183009  14.200732
  Outdoor 248206.63 123948.15   49.937486  13.139885
 Lighting 238741.28 113336.31   47.472440  12.638796
  Storage 139852.05  77730.29   55.580372   7.403670
     Bath 109218.71  51105.87   46.792230   5.781962


## 10. Top Products Analysis

In [10]:

prod_perf = (
    completed.groupby(['product_id','product_name','category'])
    .agg(revenue=('net_revenue','sum'), profit=('profit','sum'), units=('quantity','sum'))
    .reset_index()
)
prod_perf['margin_pct'] = prod_perf['profit'] / prod_perf['revenue'] * 100

top10  = prod_perf.nlargest(10, 'revenue')
bot10p = prod_perf.nsmallest(10, 'profit')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

axes[0].barh(top10['product_name'].str[:28], top10['revenue'], color=COLORS['teal'])
axes[0].set_title('Top 10 Products by Revenue', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))

colors_bot = [COLORS['coral'] if p < 0 else COLORS['amber'] for p in bot10p['profit']]
axes[1].barh(bot10p['product_name'].str[:28], bot10p['profit'], color=colors_bot)
axes[1].set_title('Lowest-Profit Products', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))
axes[1].axvline(0, color='#333', lw=0.8)

plt.tight_layout()
plt.savefig(PROC/'top_products.png', bbox_inches='tight')
plt.show()
print('Top 5 by Revenue:')
print(top10[['product_name','category','revenue','margin_pct']].head(5).to_string(index=False))


Top 5 by Revenue:
             product_name  category  revenue  margin_pct
       Cotton Patio Chair   Outdoor 67034.90   44.750511
Classic Dining Chair Pair Furniture 52017.94   42.622699
Coastal Picture Frame Set     Decor 44497.23   55.980721
            Ceramic Quilt   Bedding 41379.17   51.194284
        Bamboo Nightstand Furniture 41339.23   42.193045


## 11. Customer & Geographic Analysis

In [11]:

# Customer orders
cust = completed.groupby('customer_id').agg(
    revenue=('net_revenue','sum'),
    orders=('order_id','nunique'),
    last_purchase=('order_date','max')
).reset_index()
cust['is_repeat'] = cust['orders'] > 1

repeat_rate = cust['is_repeat'].mean() * 100
print(f'Repeat Customer Rate: {repeat_rate:.1f}%')
print(f'One-time customers:  {(~cust.is_repeat).sum():,}')
print(f'Repeat customers:    {cust.is_repeat.sum():,}')

# RFM
snapshot = completed['order_date'].max() + pd.Timedelta(days=1)
rfm = cust.copy()
rfm['recency_days'] = (snapshot - rfm['last_purchase']).dt.days

def score_q(s, asc=True):
    labels = [1,2,3,4,5] if asc else [5,4,3,2,1]
    return pd.qcut(s.rank(method='first'), 5, labels=labels).astype(int)

rfm['r_score'] = score_q(rfm['recency_days'], asc=False)
rfm['f_score'] = score_q(rfm['orders'], asc=True)
rfm['m_score'] = score_q(rfm['revenue'], asc=True)

def seg(r):
    rv, f, m = r.r_score, r.f_score, r.m_score
    if rv>=4 and f>=4 and m>=4: return 'Champions'
    if rv>=3 and f>=3:           return 'Loyal Customers'
    if rv>=4 and f<=2:           return 'Recent Customers'
    if rv>=3 and m>=3:           return 'Potential Loyalists'
    if rv<=2 and f>=3 and m>=3: return 'At Risk'
    if rv<=2 and f<=2 and m<=2: return 'Lost'
    return 'Needs Attention'

rfm['segment'] = rfm.apply(seg, axis=1)

seg_counts = rfm['segment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Pie of RFM segments
axes[0].pie(seg_counts.values, labels=seg_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('Set2', len(seg_counts)), startangle=90)
axes[0].set_title('Customer Distribution by RFM Segment', fontweight='bold')

# State revenue
state_rev = completed.groupby('state')['net_revenue'].sum().nlargest(12)
axes[1].barh(state_rev.index, state_rev.values, color=COLORS['navy'])
axes[1].set_title('Top 12 States by Revenue', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))

plt.tight_layout()
plt.savefig(PROC/'customer_geo.png', bbox_inches='tight')
plt.show()


Repeat Customer Rate: 64.4%
One-time customers:  729
Repeat customers:    1,318


## 12. Profitability & Discount Analysis

In [12]:

disc_summ = completed.groupby('has_discount').agg(
    revenue=('net_revenue','sum'), profit=('profit','sum'),
    orders=('order_id','nunique')
).reset_index()
disc_summ['margin_pct'] = disc_summ['profit'] / disc_summ['revenue'] * 100
disc_summ['label'] = disc_summ['has_discount'].map({True:'Discounted', False:'Full Price'})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(disc_summ['label'], disc_summ['revenue'],
            color=[COLORS['amber'], COLORS['teal']])
axes[0].set_title('Revenue: Full Price vs Discounted', fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))

axes[1].bar(disc_summ['label'], disc_summ['margin_pct'],
            color=[COLORS['amber'], COLORS['teal']])
axes[1].set_title('Profit Margin %: Full Price vs Discounted', fontweight='bold')
axes[1].set_ylabel('Profit Margin %')
for i, (_, row) in enumerate(disc_summ.iterrows()):
    axes[1].text(i, row.margin_pct + 0.5, f"{row.margin_pct:.1f}%", ha='center', fontweight='bold')
axes[1].set_ylim(0, 60)

plt.tight_layout()
plt.savefig(PROC/'discount_analysis.png', bbox_inches='tight')
plt.show()

print('Discount Impact Summary:')
print(disc_summ[['label','revenue','profit','orders','margin_pct']].to_string(index=False))
gap = disc_summ.loc[disc_summ.has_discount==False,'margin_pct'].values[0] - disc_summ.loc[disc_summ.has_discount==True,'margin_pct'].values[0]
print(f'\nMargin gap (Full Price vs Discounted): {gap:.1f} percentage points')


Discount Impact Summary:
     label    revenue    profit  orders  margin_pct
Full Price 1656280.96 834171.42    7166   50.364125
Discounted  232674.87  94911.94    1290   40.791659

Margin gap (Full Price vs Discounted): 9.6 percentage points


## 13. Seasonality Analysis

In [13]:

monthly_avg = (
    completed.groupby(['order_year','order_month_num','order_month_name'])
    ['net_revenue'].sum().reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for yr, grp in monthly_avg.groupby('order_year'):
    g = grp.sort_values('order_month_num')
    axes[0].plot(g['order_month_num'], g['net_revenue'],
                 marker='o', label=str(yr), linewidth=2)
axes[0].set_title('Monthly Revenue by Year (Seasonality)', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Net Revenue ($)')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))
axes[0].legend()

# Quarterly
qtr = completed.groupby(['order_year','order_quarter'])['net_revenue'].sum().reset_index()
qtr['label'] = qtr['order_year'].astype(str) + ' Q' + qtr['order_quarter'].astype(str)
axes[1].bar(qtr['label'], qtr['net_revenue'],
            color=[COLORS['teal'] if str(y)=='2024' else COLORS['navy'] for y in qtr['order_year']])
axes[1].set_title('Quarterly Revenue', fontweight='bold')
axes[1].set_xticklabels(qtr['label'], rotation=30)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'${v/1e3:.0f}K'))

plt.tight_layout()
plt.savefig(PROC/'seasonality.png', bbox_inches='tight')
plt.show()


## 14. Business Insights & Recommendations

### Finding 1 — Revenue & Profitability are Healthy
**Metric:** $1.89M revenue, 49.2% margin over 24 months  
**Implication:** The business is profitable across all 8 product categories  
**Action:** Maintain pricing discipline, especially in Furniture (lowest margin at 46.2%)  
**Monitor:** Monthly profit margin % per category

### Finding 2 — Discounts Cost 9.4 Margin Points
**Metric:** Full-price margin = 50.4%; Discounted margin = 40.8%  
**Implication:** 1,290 discounted orders (15.3% of total) dilute overall profitability  
**Action:** Limit discount codes to Champions and Loyal Customers RFM segments only  
**Monitor:** Discount Rate %, Margin on Discounted Orders

### Finding 3 — At Risk Customers Need Immediate Action
**Metric:** 405 customers classified as At Risk (high F+M, low R)  
**Implication:** These were once frequent buyers — losing them costs more than acquiring new ones  
**Action:** Automated win-back email at 60–90 days since last purchase  
**Monitor:** At Risk → Loyal conversion rate

### Finding 4 — Storage is the Margin Champion
**Metric:** Storage margin = 55.6% but only 7.4% revenue share  
**Implication:** Significant profit upside if Storage category grows  
**Action:** Expand Storage SKUs from ~10 to 20+; increase ad budget allocation  
**Monitor:** Storage Revenue Share %, Storage SKU count

### Finding 5 — Geographic Concentration
**Metric:** CA + NY + TX = 29.3% of revenue; 20+ states each under $10K  
**Implication:** Heavy dependence on 3 coastal states  
**Action:** Run targeted paid campaigns in FL and PA (already $84K–$142K each with growth potential)  
**Monitor:** Revenue per State, New Customer Acquisition by State

### Finding 6 — Organic Search is Underinvested
**Metric:** Organic = $777K revenue (41%) from 741 customers; Paid Social = $239K from 367 customers  
**Implication:** Organic generates more revenue per customer than Paid Social  
**Action:** Invest in SEO content marketing; blog/guide content for home goods  
**Monitor:** Organic Search Revenue, Organic Traffic (Google Analytics)


## 15. Power BI Export Validation

In [14]:

import os

pbi_files = list(PBI.glob('*.csv'))
print(f'Power BI export files: {len(pbi_files)}')
for f in sorted(pbi_files):
    df_tmp = pd.read_csv(f)
    print(f'  {f.name:<35} {len(df_tmp):>8,} rows | {len(df_tmp.columns):>3} cols')

# Revenue reconciliation
fs = pd.read_csv(PBI/'FactSales.csv')
completed_check = fs[fs['IsCompleted']==True]
notebook_revenue = completed['net_revenue'].sum()
export_revenue   = completed_check['NetRevenue'].sum()
diff = abs(notebook_revenue - export_revenue)

print(f'\nRevenue Reconciliation:')
print(f'  Notebook total:    ${notebook_revenue:,.2f}')
print(f'  FactSales export:  ${export_revenue:,.2f}')
print(f'  Difference:        ${diff:.2f}')
print(f'  Status: {"✓ MATCH" if diff < 0.01 else "✗ MISMATCH — investigate"}')

# FK integrity
dc = pd.read_csv(PBI/'DimCustomer.csv')
dp = pd.read_csv(PBI/'DimProduct.csv')
fs_cust = set(fs.CustomerID.dropna().astype(int))
dc_cust = set(dc.CustomerID.astype(int))
print(f'\nCustomer FK coverage: {len(fs_cust & dc_cust)}/{len(fs_cust)} ({len(fs_cust & dc_cust)/len(fs_cust)*100:.1f}%)')
fs_prod = set(fs.ProductID.dropna().astype(int))
dp_prod = set(dp.ProductID.astype(int))
print(f'Product  FK coverage: {len(fs_prod & dp_prod)}/{len(fs_prod)} ({len(fs_prod & dp_prod)/len(fs_prod)*100:.1f}%)')


Power BI export files: 7
  BusinessInsights.csv                       8 rows |   9 cols
  CustomerRFM.csv                        2,047 rows |  12 cols
  DataQualityReport.csv                      7 rows |   5 cols
  DimCustomer.csv                        3,600 rows |   9 cols
  DimDate.csv                              731 rows |  13 cols
  DimProduct.csv                            90 rows |   6 cols
  FactSales.csv                         15,668 rows |  29 cols

Revenue Reconciliation:
  Notebook total:    $1,888,955.83
  FactSales export:  $1,888,955.83
  Difference:        $0.00
  Status: ✓ MATCH

Customer FK coverage: 2106/2106 (100.0%)
Product  FK coverage: 90/90 (100.0%)


## 16. Limitations & Future Improvements

### Current Limitations
1. **No sales targets** — variance analysis requires a monthly target file
2. **No shipping dates** — shipping duration analysis not possible
3. **No product sub-categories** — limited product drill-down granularity
4. **No COGS beyond product cost** — operating expenses not modelled
5. **60-row fact table over-count** — caused by 60 customers with duplicate signup dates in source; not affecting revenue totals
6. **Profit reflects gross margin only** — marketing, shipping, and overhead excluded

### Recommended Next Steps
1. Upload a monthly sales target CSV (columns: year_month, category, target_sales)
2. Add ship_date to orders for fulfilment analysis
3. Connect Google Analytics for true multi-touch attribution by channel
4. Run this analysis quarterly to track RFM segment migration
5. Add A/B test data for discount code experiments
